In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml kailash-align

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# M6 Ollama bootstrap — local LLMs, no API keys
# On Colab: installs + starts Ollama and pulls this lesson's models.
# Locally: verifies the daemon is running and models are pulled.
# ══════════════════════════════════════════════════════════════════
_LESSON_MODELS = ['qwen2.5:0.5b', 'llama3.2:3b']
import sys, os, time, shutil, subprocess
import httpx

_OLLAMA_BASE_URL = os.environ.get('OLLAMA_BASE_URL', 'http://localhost:11434')
_IN_COLAB = 'google.colab' in sys.modules

def _daemon_up(timeout_s=1.0):
    try:
        r = httpx.get(f'{_OLLAMA_BASE_URL}/api/tags', timeout=timeout_s)
        r.raise_for_status()
        return r.json()
    except Exception:
        return None

if _IN_COLAB and shutil.which('ollama') is None:
    print('[ollama] installing binary …')
    subprocess.run(
        'curl -fsSL https://ollama.com/install.sh | sh',
        shell=True, check=True,
    )

if _IN_COLAB and _daemon_up() is None:
    print('[ollama] starting daemon …')
    subprocess.Popen(
        'nohup ollama serve > /tmp/ollama.log 2>&1 &',
        shell=True,
    )
    deadline = time.monotonic() + 30
    while time.monotonic() < deadline and _daemon_up() is None:
        time.sleep(0.5)

_tags = _daemon_up(timeout_s=2.0)
if _tags is None:
    raise RuntimeError(
        f'Ollama daemon not reachable at {_OLLAMA_BASE_URL}. '
        'On Colab: re-run this cell. Locally: run `ollama serve`.'
    )

_have = {m.get('name','').split(':',1)[0] for m in _tags.get('models', [])}
for _model in _LESSON_MODELS:
    _fam = _model.split(':', 1)[0]
    if _fam in _have:
        continue
    if _IN_COLAB:
        print(f'[ollama] pulling {_model} (one-time) …')
        subprocess.run(['ollama', 'pull', _model], check=True)
    else:
        raise RuntimeError(
            f'Model {_model!r} not pulled. Run: ollama pull {_model}'
        )

print(
    '✓ Ollama ready at', _OLLAMA_BASE_URL,
    '— models:', _LESSON_MODELS,
)



In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp06
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


def create_visualizer():
    """Return a ModelVisualizer with the P2 experimental notice acknowledged.

    kailash-ml 2.2.2 emits ExperimentalWarning (a UserWarning) at
    ModelVisualizer construction; exercises run under warnings-as-errors,
    so the notice is acknowledged narrowly here — the ONE construction site
    every course file should use.
    """
    import warnings as _warnings

    from kailash_ml import ModelVisualizer
    from kailash_ml._decorators import ExperimentalWarning as _EW

    with _warnings.catch_warnings():
        _warnings.simplefilter("ignore", _EW)
        return ModelVisualizer()

# ── SDK-internal deprecation nag filter (strict-gate hygiene) ────────────
# torch 2.12's ONNX exporter warns that dynamic_axes is "not recommended"
# under dynamo=True — a deprecation notice inside OnnxBridge's internals,
# not a correctness signal, and fatal under warnings-as-errors. Acknowledged
# here (kailash_helpers is imported by every shared.mlfpNN package, so every
# technique file gets it); message-matched, everything else still errors.
import warnings as _warnings

_warnings.filterwarnings(
    "ignore", message=r".*dynamic_axes.*not recommended.*", category=UserWarning
)
# matplotlib's Agg backend (forced in headless/script runs) nags
# "FigureCanvasAgg is non-interactive, and thus cannot be shown" on every
# plt.show() — a backend notice, not a correctness signal.
_warnings.filterwarnings(
    "ignore", message=r".*non-interactive.*cannot be shown.*", category=UserWarning
)

def hdb_storey_range_expr(column: str = "storey_range"):
    """Polars expression normalising HDB storey_range letter-O typos.

    The raw HDB resale file has letter-O typos ("O4 TO 06", "1O TO 12",
    "28 TO 3O"); read a letter O next to a digit as zero, leaving "TO"
    alone. Single home for the rule (P2): M2 ex_8 and M3 ex_5/07 both
    normalise at the use-site.
    """
    import polars as pl

    return (
        pl.col(column)
        .str.replace_all(r"\bO(\d)", "0${1}")
        .str.replace_all(r"(\d)O\b", "${1}0")
    )



# ── shared/mlfp06/ex_2.py ──
"""
Shared infrastructure for Exercise 2 — LLM Fine-Tuning.

Contains: SFT dataset loader (IMDB -> instruction pairs), LoRA/adapter
parameter-counting helpers, kailash-align config factory, base
hyperparameters, and output directory. Technique-specific classes
(LoRALayer, AdapterLayer, etc.) live in the per-technique files.
"""

import os
from pathlib import Path

import polars as pl
import torch


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
device = get_device()

# Output directory for all visualisation and training artifacts
REPO_ROOT = Path.cwd()
OUTPUT_DIR = REPO_ROOT / "outputs" / "ex2_finetuning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# SHARED HYPERPARAMETERS
# ════════════════════════════════════════════════════════════════════════

D_MODEL = 512  # synthetic demo hidden dim for from-scratch LoRA / adapter
LORA_RANK = 8  # default LoRA rank for the demos
LORA_ALPHA = 16.0  # typical alpha = 2 * rank
ADAPTER_BOTTLENECK = 64  # default adapter bottleneck

# ════════════════════════════════════════════════════════════════════════
# SFT DATA LOADING — IMDB instruction-response pairs
# ════════════════════════════════════════════════════════════════════════

DATA_DIR = REPO_ROOT / "data" / "mlfp06" / "imdb"
DATA_DIR.mkdir(parents=True, exist_ok=True)
CACHE_FILE = DATA_DIR / "imdb_sft_2k.parquet"


def load_imdb_sft(
    n_rows: int = 2000, train_frac: float = 0.9
) -> tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame]:
    """Load IMDB movie reviews and reformat as SFT instruction pairs.

    Returns:
        (full, train, eval) polars DataFrames with columns:
            instruction, response, text, label
    """
    if CACHE_FILE.exists():
        print(f"Loading cached IMDB SFT pairs from {CACHE_FILE}")
        sft_data = pl.read_parquet(CACHE_FILE)
    else:
        print("Downloading stanfordnlp/imdb from HuggingFace (first run)...")
        from datasets import load_dataset

        ds = load_dataset("stanfordnlp/imdb", split="train")
        ds = ds.shuffle(seed=42).select(range(min(n_rows, len(ds))))

        label_names = {0: "negative", 1: "positive"}
        rows = []
        for row in ds:
            review = row["text"][:1500]
            sentiment = label_names[row["label"]]
            rows.append(
                {
                    "instruction": (
                        "Classify the sentiment of the following movie review as "
                        "either 'positive' or 'negative', then briefly justify "
                        f"your answer.\n\nReview: {review}"
                    ),
                    "response": (
                        f"Sentiment: {sentiment}. The reviewer expresses a clearly "
                        f"{sentiment} reaction to the film."
                    ),
                    "text": review,
                    "label": sentiment,
                }
            )
        sft_data = pl.DataFrame(rows)
        sft_data.write_parquet(CACHE_FILE)
        print(f"Cached {sft_data.height} SFT pairs to {CACHE_FILE}")

    n_train = int(sft_data.height * train_frac)
    train_data = sft_data[:n_train]
    eval_data = sft_data[n_train:]
    print(
        f"IMDB SFT loaded: {sft_data.height} pairs "
        f"({train_data.height} train / {eval_data.height} eval)"
    )
    return sft_data, train_data, eval_data


# ════════════════════════════════════════════════════════════════════════
# PARAMETER COUNTING HELPERS
# ════════════════════════════════════════════════════════════════════════


def count_lora_params(
    d_model: int, rank: int, num_modules: int = 1, num_layers: int = 1
) -> int:
    """LoRA trainable params = 2 * d * r per target module per layer.

    For d=512, r=8, modules=1, layers=1 -> 8,192 params.
    """
    return (d_model * rank + rank * d_model) * num_modules * num_layers


def count_adapter_params(d_model: int, bottleneck: int, num_layers: int = 1) -> int:
    """Adapter params per layer: down(d*b+b) + up(b*d+d) + layernorm(2*d).

    Matches the AdapterLayer in 02_adapter_from_scratch.py.
    """
    return (
        d_model * bottleneck + bottleneck + bottleneck * d_model + d_model + 2 * d_model
    ) * num_layers


def full_finetune_params(
    d_model: int, num_modules: int = 1, num_layers: int = 1
) -> int:
    """Full fine-tuning baseline: d^2 per module per layer."""
    return d_model * d_model * num_modules * num_layers


# ════════════════════════════════════════════════════════════════════════
# KAILASH-ALIGN CONFIG FACTORY
# ════════════════════════════════════════════════════════════════════════


def get_base_model_name() -> str:
    """Return the HuggingFace repo id of the SFT base model.

    Resolution: ``SFT_BASE_MODEL`` env var → ``Qwen/Qwen2.5-0.5B-Instruct``.
    Note: Align/TRL training needs a HuggingFace repo id, NOT an Ollama tag —
    the Ollama-side ``OLLAMA_FT_BASE_MODEL`` is for inference of the
    fine-tuned model after GGUF export, not for training.
    """
    return os.environ.get("SFT_BASE_MODEL") or "Qwen/Qwen2.5-0.5B-Instruct"


def build_sft_config(
    base_model: str | None = None,
    lora_r: int = 16,
    lora_alpha: int = 32,
    num_epochs: int = 3,
    output_subdir: str = "sft_output",
):
    """Build a kailash-align AlignmentConfig for SFT+LoRA training.

    Imported lazily so technique files that do not call SFT are not
    forced to install the align extra at import time.

    kailash-align 0.6.0+ uses a composed AlignmentConfig: top-level
    method + base_model_id + experiment_dir, plus required LoRAConfig
    + SFTConfig + DPOConfig sub-configs. The DPOConfig is unused for
    SFT but the constructor still expects it (default values are fine).
    """
    from kailash_align import AlignmentConfig, DPOConfig, LoRAConfig, SFTConfig

    return AlignmentConfig(
        method="sft",
        base_model_id=base_model or get_base_model_name(),
        lora=LoRAConfig(
            rank=lora_r,
            alpha=lora_alpha,
            target_modules=("q_proj", "v_proj"),
            dropout=0.05,
        ),
        sft=SFTConfig(
            num_train_epochs=num_epochs,
            per_device_train_batch_size=4,
            gradient_accumulation_steps=4,
            learning_rate=2e-4,
            warmup_ratio=0.1,
            max_seq_length=512,
        ),
        dpo=DPOConfig(),
        experiment_dir=str(OUTPUT_DIR / output_subdir),
    )




# ════════════════════════════════════════════════════════════════════════
# MLFP06 — Exercise 2.4: Model Merging Without Training
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - THEORY: why merging fine-tuned models works (task vectors)
#   - BUILD: TIES (Trim, Elect Sign, Merge) on synthetic deltas
#   - BUILD: SLERP (spherical linear interpolation) on weight tensors
#   - VISUALISE: weight-norm preservation across merge strategies
#   - APPLY: Singapore fintech — three LoRAs into one deployment
#
# PREREQUISITES: Exercises 2.1 and 2.2 (LoRA + adapters)
# ESTIMATED TIME: ~30 min
#
# TASKS:
#   1. THEORY: task vectors and the merge landscape
#   2. BUILD: TIES merge (trim -> elect sign -> average)
#   3. BUILD: SLERP merge (hypersphere interpolation)
#   4. VISUALISE: norm comparison across merge strategies
#   5. APPLY: Singapore fintech KYC + fraud + explainer merge
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import polars as pl
import torch
from dotenv import load_dotenv


load_dotenv()

torch.manual_seed(42)



## THEORY — Task Vectors and the Merge Landscape


In [ ]:
# A task vector is the *difference* between a fine-tuned model and its
# base:  tau = W_finetuned - W_base.  It captures everything the
# fine-tuning added, and nothing else.
#
# Task arithmetic (Ilharco et al., 2023):
#   - Add:      W_merged = W_base + alpha * tau_A + beta * tau_B
#   - Subtract: W_merged = W_base - alpha * tau_A    (remove a skill)
#   - Negate:   reverses the adaptation entirely
#
# But raw addition of task vectors causes *sign conflict*: two fine-tunes
# may update the same parameter in opposite directions, and averaging
# cancels both out into noise.  TIES and DARE solve this.
#
# TIES (Yadav et al., 2023), three steps per parameter:
#   TRIM  — keep only each task vector's top-k% largest-magnitude entries
#           (k = "density", 20% in the paper); the rest is treated as noise
#   ELECT — the merged sign is the sign of the SUMMED trimmed deltas, so
#           the direction with more total mass wins a conflict (it is not
#           zeroed out)
#   MERGE — average only the non-zero deltas that agree with that sign
#           (a "disjoint mean": tasks that were trimmed or disagree there
#           do not dilute the average)
#
# DARE (Yu et al., 2023): randomly DROP a fraction of delta parameters
# then RESCALE the survivors by 1/(1-drop).  Like dropout for merging.
#
# SLERP (Spherical Linear Interpolation): walks the shortest path on the
# hypersphere between two weight tensors rather than the straight chord.
# Preserves weight norms, which matters because transformer weights are
# calibrated to a specific norm at training time.
#
# Crucially: merging is FREE — zero extra training compute.  This makes
# it the cheapest way to combine capabilities from multiple fine-tunes.



## TASK 1 — BUILD: TIES merge on synthetic 128x128 deltas


Step 1 — TRIM: keep the top `density` fraction of |delta|, zero the rest.


In [ ]:
print("\n" + "=" * 70)
print("TASK 1: TIES merge — trim / elect sign / average")
print("=" * 70)

delta_A = torch.randn(128, 128) * 0.1  # Task A fine-tuned delta
delta_B = torch.randn(128, 128) * 0.1  # Task B fine-tuned delta

TRIM_DENSITY = 0.20  # keep the top 20% of each task vector by magnitude


def trim_top_k(delta: torch.Tensor, density: float) -> torch.Tensor:
    k = max(1, int(delta.numel() * density))
    threshold = delta.abs().flatten().topk(k).values.min()
    return torch.where(delta.abs() >= threshold, delta, torch.zeros_like(delta))


delta_A_trim = trim_top_k(delta_A, TRIM_DENSITY)
delta_B_trim = trim_top_k(delta_B, TRIM_DENSITY)

# Step 2 — ELECT SIGN: sign of the summed (mass-weighted) trimmed deltas.
# On a conflict the larger-magnitude update wins instead of both cancelling.
sign_A = delta_A_trim.sign()
sign_B = delta_B_trim.sign()
elected_sign = (delta_A_trim + delta_B_trim).sign()

# Step 3 — DISJOINT MERGE: average only the non-zero deltas whose sign
# agrees with the elected sign.
mask_A = ((sign_A == elected_sign) & (delta_A_trim != 0)).float()
mask_B = ((sign_B == elected_sign) & (delta_B_trim != 0)).float()
merged_delta = (delta_A_trim * mask_A + delta_B_trim * mask_B) / (
    mask_A + mask_B
).clamp(min=1.0)

both_kept = (delta_A_trim != 0) & (delta_B_trim != 0)
conflicts = both_kept & (sign_A != sign_B)
print(f"Original non-zero params (delta_A): {(delta_A != 0).sum().item():,}")
print(f"After TRIM (top {TRIM_DENSITY:.0%}):               {(delta_A_trim != 0).sum().item():,}")
print(f"Entries kept by BOTH tasks:         {both_kept.sum().item():,}")
print(f"  ...of which sign conflicts:       {conflicts.sum().item():,}")
print(f"Merged delta Frobenius norm:        {merged_delta.norm():.4f}")



### Checkpoint 1


In [ ]:
assert merged_delta.shape == delta_A.shape, "Task 1: TIES should preserve shape"
assert (delta_A_trim != 0).sum() < (delta_A != 0).sum(), "TRIM should zero some values"
print("✓ Checkpoint 1 passed — TIES merge complete\n")

# INTERPRETATION: TIES prevents sign cancellation during merging.
# With plain averaging, two task vectors that disagree on a parameter
# cancel each other there. TIES keeps the update with more mass for each
# conflicting entry and averages only agreeing, non-trimmed values.
# (These are random synthetic deltas, so roughly half of the overlapping
# entries conflict; real task vectors from one base usually agree more.)



## TASK 2 — BUILD: SLERP merge on weight tensors


t=0 -> v0; t=1 -> v1; t=0.5 -> midpoint on the hypersphere.
    Falls back to linear interpolation when the vectors are nearly
    parallel (theta < 1e-6) to avoid divide-by-zero.


In [ ]:
def slerp(t: float, v0: torch.Tensor, v1: torch.Tensor) -> torch.Tensor:
    v0_flat = v0.flatten().float()
    v1_flat = v1.flatten().float()
    cos_theta = torch.dot(v0_flat, v1_flat) / (v0_flat.norm() * v1_flat.norm() + 1e-8)
    cos_theta = cos_theta.clamp(-1.0, 1.0)
    theta = torch.acos(cos_theta)
    if theta.abs() < 1e-6:
        return (1 - t) * v0 + t * v1
    sin_theta = torch.sin(theta)
    w0 = torch.sin((1 - t) * theta) / sin_theta
    w1 = torch.sin(t * theta) / sin_theta
    return w0 * v0 + w1 * v1


print("=" * 70)
print("TASK 2: SLERP vs linear interpolation — weight norm comparison")
print("=" * 70)

W_base = torch.randn(128, 128)
W_task_A = W_base + delta_A
W_task_B = W_base + delta_B

slerp_merged = slerp(0.5, W_task_A, W_task_B)
linear_merged = 0.5 * W_task_A + 0.5 * W_task_B

norms = {
    "W_task_A": W_task_A.norm().item(),
    "W_task_B": W_task_B.norm().item(),
    "SLERP midpoint": slerp_merged.norm().item(),
    "Linear midpoint": linear_merged.norm().item(),
}
for k, v in norms.items():
    print(f"  {k:<20} norm = {v:.4f}")

mean_input_norm = 0.5 * (norms["W_task_A"] + norms["W_task_B"])
slerp_gap = abs(norms["SLERP midpoint"] - mean_input_norm)
linear_gap = abs(norms["Linear midpoint"] - mean_input_norm)

print(f"\n  SLERP deviation from mean input norm:  {slerp_gap:.4f}")
print(f"  Linear deviation from mean input norm: {linear_gap:.4f}")



### Checkpoint 2


In [ ]:
assert slerp_merged.shape == W_task_A.shape, "Task 2: SLERP should preserve shape"
assert slerp_gap <= linear_gap + 1e-6, "SLERP should preserve norms at least as well"
print("✓ Checkpoint 2 passed — SLERP preserves norms\n")

# INTERPRETATION: Linear merging shrinks toward the origin whenever the
# two inputs are not perfectly aligned.  SLERP walks the arc on the
# hypersphere and keeps the interpolated norm close to the inputs.



## TASK 3 — VISUALISE: norm comparison across merge strategies


In [ ]:
print("=" * 70)
print("TASK 3: Visualise merge-strategy norm preservation")
print("=" * 70)

# Sweep the interpolation parameter t from 0 -> 1 and record norms
ts = [i / 10 for i in range(11)]
slerp_norms = [slerp(t, W_task_A, W_task_B).norm().item() for t in ts]
linear_norms = [((1 - t) * W_task_A + t * W_task_B).norm().item() for t in ts]

fname = OUTPUT_DIR / "ex2_slerp_vs_linear.png"
fname.unlink(missing_ok=True)  # the checkpoint must see THIS run's plot

fig, ax = plt.subplots(1, 1, figsize=(9, 5))
ax.plot(ts, slerp_norms, "o-", color="steelblue", linewidth=2, label="SLERP")
ax.plot(ts, linear_norms, "s-", color="darkorange", linewidth=2, label="Linear")
ax.axhline(
    norms["W_task_A"], color="gray", linestyle=":", alpha=0.6, label="input norms"
)
ax.axhline(norms["W_task_B"], color="gray", linestyle=":", alpha=0.6)
ax.set_xlabel("Interpolation parameter t (0 = task A, 1 = task B)")
ax.set_ylabel("Frobenius norm of merged weights")
ax.set_title("SLERP vs Linear — weight norm preservation", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(fname, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"  Saved: {fname}")



### Checkpoint 3


In [ ]:
assert fname.exists(), "Task 3: plot should exist"
print("✓ Checkpoint 3 passed — norm comparison visualised\n")



## TASK 4 — APPLY: Singapore fintech — merging three LoRAs into one


In [ ]:
# SCENARIO (illustrative): A Singapore digital bank has trained three independent LoRA
# adapters on top of the same 7B base:
#   tau_kyc     — KYC document extraction (NRIC, addresses, phone formats)
#   tau_fraud   — Transaction-narrative fraud flagging
#   tau_explain — Plain-English explanations for rejected applications
#
# Each adapter was trained by a different team over several weeks.  The
# platform team wants to ship a SINGLE deployed model that handles all
# three tasks instead of three separate inference pipelines.
#
# OPTIONS:
#   A. Three deployments (status quo): 3 * 14 GB GPU VRAM per region.
#      High latency for multi-step journeys (KYC -> fraud -> explain).
#   B. Train a multi-task adapter: ~2 weeks of additional GPU time to
#      re-train on pooled data, new evals needed for each capability.
#   C. TIES merge the three LoRAs into one: zero training, ~2 hours of
#      engineering to run the merge + eval.  Some quality drop on each
#      task but typically <2 percentage points.
#
# DECISION: TIES merge (option C).  The $/capability trade-off favours
# free merging unless the merged model drops >5 points on any eval.
#
# BUSINESS IMPACT (illustrative figures):
#   - Infra saving: drop from 3 * 14 GB to 1 * 14 GB per region (28 GB
#     freed in each of 3 regions).  At ~S$1.20/GB-month on managed GPU,
#     that is ~S$33/region/month * 3 = S$99/month of inference VRAM.
#   - Latency: end-to-end KYC -> fraud -> explain journey drops from
#     ~780 ms (three hops) to ~320 ms (single inference).  On the
#     bank's 180,000 onboarding journeys/month, that saves ~23 agent-
#     hours of wait-time monitoring, or ~S$4,100/month in operations
#     cost.
#   - Avoided retraining: option B would have consumed ~S$18,000 of
#     GPU time plus ~6 weeks of engineering effort.
#
# RISK: merged model degrades on one of the tasks (most likely fraud,
# which is the most narrowly-scoped).  Mitigation: re-run the three
# task evaluation suites and fall back to option A if any eval drops
# below the SLA threshold.

print("Singapore fintech — three-LoRA merge decision:")
rows = [
    ("Option A: three deployments", 3 * 14, 780, 0),
    ("Option B: retrain multi-task", 1 * 14, 320, 18_000),
    ("Option C: TIES merge", 1 * 14, 320, 120),
]
decision_df = pl.DataFrame(
    {
        "Option": [r[0] for r in rows],
        "VRAM_GB": [r[1] for r in rows],
        "Latency_ms": [r[2] for r in rows],
        "Upfront_cost_SGD": [r[3] for r in rows],
    }
)
print(decision_df)
monthly_saving_sgd = 99 + 4_100
annual_saving_sgd = monthly_saving_sgd * 12
print(f"\n  Monthly saving (option C vs A): S${monthly_saving_sgd:,}")
print(f"  Annual saving:                  S${annual_saving_sgd:,}")
print(f"  Recommended: TIES merge (option C) with per-task eval gate")



### Checkpoint 4


In [ ]:
assert annual_saving_sgd > 0, "Task 4: fintech should see positive savings"
print("✓ Checkpoint 4 passed — fintech merge decision analysed\n")



## REFLECTION


[x] Built TIES merge: trim -> elect sign -> merge
  [x] Built SLERP merge: spherical interpolation on the hypersphere
  [x] Visualised that SLERP preserves weight norms better than linear
  [x] Applied merging to a Singapore fintech three-LoRA scenario
      (~S$50k/year saving vs separate deployments, zero retraining)

  KEY INSIGHT: merging is free compute.  TIES handles sign conflict,
  SLERP handles norm drift, and task arithmetic composes them for
  targeted capability addition or removal.

  Next: 05_quantisation.py surveys how we shrink the merged model for
  deployment on smaller hardware.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

